# Calidad de datos · Validaciones

## Propósito

Ejecutar las reglas activas del catálogo de calidad (`md_dq_reglas`) para un momento del pipeline, guardar el resultado de cada una y aplicar la decisión definida: detener, aislar en cuarentena o advertir.

Un solo notebook sirve a las dos tasks del Workflow. El parámetro `momento` define qué reglas corre:

| Task | `momento` | Reglas |
|---|---|---|
| `dq_validaciones_silver` | `silver` | Las que validan Silver, tras la última task Silver |
| `dq_validaciones_gold` | `gold` | Las que validan Gold, tras `gld_order_summary` |

## Flujo

1. Registrar el inicio de la ejecución en `hd_etl_log`.
2. Leer las reglas activas del momento.
3. Medir cada regla: filas evaluadas, filas que incumplen y porcentaje de cumplimiento.
4. En las reglas de cuarentena que tienen sentencia de aislamiento (integridad referencial), aislar las filas que incumplen y registrar el conteo por motivo en `hd_cuarentena_resumen`.
5. Guardar todos los resultados en `hd_dq_resultados`.
6. Enviar a Slack un aviso con las reglas que superaron su umbral.
7. Si alguna regla de DETENER incumple, o alguna regla falló al ejecutarse, detener la task. Si no, cerrar el registro como exitoso.

## Decisión por regla

El porcentaje de cumplimiento se compara con el umbral de la regla.

| Decisión | Si hay filas que incumplen | Qué pasa |
|---|---|---|
| CUARENTENA | siempre | Las filas se aíslan con su motivo y Gold no las lee. Se avisa por Slack solo si el cumplimiento queda bajo el umbral. |
| ADVERTIR | bajo el umbral | Se avisa por Slack. El pipeline sigue. |
| DETENER | bajo el umbral | Se guardan todos los resultados, se avisa y la task falla. |

Una regla que no se pudo ejecutar (consulta con error) también detiene la task: una validación rota no puede dejar pasar los datos en silencio.

## Origen y destino

* Catálogo: `metadata_dataco.supply_chain.md_dq_reglas`.
* Resultados: `metadata_dataco.supply_chain.hd_dq_resultados` (una fila por ejecución y regla).
* Cuarentena: las sentencias de las reglas de integridad referencial actualizan `estado_calidad` y `motivo_cuarentena` en la tabla Silver y suman su conteo a `hd_cuarentena_resumen`.

## Reintentos

Es seguro repetir la task: los resultados y los conteos de cuarentena de la misma ejecución se reemplazan, no se duplican.

In [0]:
# ============================================================
# Parámetros y marcadores de las consultas
# ============================================================

dbutils.widgets.dropdown("momento", "silver", ["silver", "gold"], "Momento (qué reglas corre)")

dbutils.widgets.text("silver_catalog", "slv_dataco", "Catálogo Silver")
dbutils.widgets.text("silver_schema", "supply_chain", "Esquema Silver")
dbutils.widgets.text("gold_catalog", "gld_dataco", "Catálogo Gold")
dbutils.widgets.text("gold_schema", "supply_chain", "Esquema Gold")
dbutils.widgets.text("bronze_catalog", "brz_dataco", "Catálogo Bronze")
dbutils.widgets.text("bronze_schema", "sharepoint", "Esquema Bronze")
dbutils.widgets.text("bronze_table", "raw_supply_chain", "Tabla Bronze")
dbutils.widgets.text("metadata_catalog", "metadata_dataco", "Catálogo Metadata")
dbutils.widgets.text("metadata_schema", "supply_chain", "Esquema Metadata")

momento = dbutils.widgets.get("momento")
metadata_catalog = dbutils.widgets.get("metadata_catalog")
metadata_schema = dbutils.widgets.get("metadata_schema")

TABLA_REGLAS = f"{metadata_catalog}.{metadata_schema}.md_dq_reglas"
TABLA_RESULTADOS = f"{metadata_catalog}.{metadata_schema}.hd_dq_resultados"
TABLA_CUARENTENA_RESUMEN = f"{metadata_catalog}.{metadata_schema}.hd_cuarentena_resumen"

MARCADORES = {
    "{silver}": f"{dbutils.widgets.get('silver_catalog')}.{dbutils.widgets.get('silver_schema')}",
    "{gold}": f"{dbutils.widgets.get('gold_catalog')}.{dbutils.widgets.get('gold_schema')}",
    "{control}": f"{metadata_catalog}.{metadata_schema}",
    "{bronze}": f"{dbutils.widgets.get('bronze_catalog')}.{dbutils.widgets.get('bronze_schema')}.{dbutils.widgets.get('bronze_table')}",
}


def resolver(consulta: str) -> str:
    for marcador, valor in MARCADORES.items():
        consulta = consulta.replace(marcador, valor)
    return consulta


print(f"✓ Momento: {momento}")
print(f"✓ Catálogo de reglas: {TABLA_REGLAS}")

In [0]:
# ============================================================
# INICIO DE LOG DE EJECUCIÓN — dq_validaciones
# ============================================================

import uuid

dbutils.widgets.text("id_ejecucion_manual", "", "Id de ejecución (solo corridas manuales)")

TABLA_ETL_LOG = f"{metadata_catalog}.{metadata_schema}.hd_etl_log"

id_manual = dbutils.widgets.get("id_ejecucion_manual")
try:
    id_ejecucion = dbutils.jobs.taskValues.get(taskKey="prepare_pipeline_control", key="id_ejecucion")
except Exception:
    if not id_manual:
        raise
    id_ejecucion = id_manual

LOG_ID = str(uuid.uuid4())
NOTEBOOK = f"dq_validaciones_{momento}"
CAPA = "calidad"

spark.sql(f"""
INSERT INTO {TABLA_ETL_LOG}
(log_id, id_ejecucion, notebook, capa, estado, fecha_inicio)
VALUES ('{LOG_ID}', '{id_ejecucion}', '{NOTEBOOK}', '{CAPA}', 'EN_PROCESO', current_timestamp())
""")

try:
    dbutils.jobs.taskValues.set(key="log_id", value=LOG_ID)
    print("✓ log_id publicado como Task Value")
except Exception:
    if not id_manual:
        raise
    print("• Corrida manual: log_id no se publica como Task Value")

print(f"✓ Registro EN_PROCESO creado en etl_log: {LOG_ID}")

In [0]:
# ============================================================
# Decisión de una regla
# ------------------------------------------------------------
# Devuelve (resultado, avisar):
#   resultado: CUMPLE, CUARENTENA, ADVERTENCIA, DETENIDO o ERROR
#   avisar:    True si debe llegar un aviso a Slack
# ============================================================

def decidir(incumplen, porcentaje, umbral, decision):
    if incumplen == 0:
        return "CUMPLE", False

    bajo_umbral = porcentaje < umbral

    if decision == "CUARENTENA":
        return "CUARENTENA", bajo_umbral
    if decision == "ADVERTIR":
        return ("ADVERTENCIA", True) if bajo_umbral else ("CUMPLE", False)
    if decision == "DETENER":
        return ("DETENIDO", True) if bajo_umbral else ("CUMPLE", False)
    return "ERROR", True


def porcentaje_cumplimiento(evaluadas, incumplen):
    if evaluadas == 0:
        return 100.0
    return round(100.0 * (evaluadas - incumplen) / evaluadas, 3)

In [0]:
# ============================================================
# Lectura de las reglas activas del momento
# ============================================================

from pyspark.sql import functions as F

reglas = (
    spark.table(TABLA_REGLAS)
    .filter((F.col("estado") == "ACTIVA") & (F.col("momento_ejecucion") == momento))
    .orderBy("regla_id")
    .collect()
)

print(f"✓ Reglas activas para el momento '{momento}': {len(reglas)}")
if not reglas:
    print("⚠️ No hay reglas activas para este momento: no se valida nada")

In [0]:
# ============================================================
# Medición de cada regla y aislamiento de las filas que incumplen
# ------------------------------------------------------------
# 1) Cuenta filas evaluadas y filas que incumplen.
# 2) Si la decisión es CUARENTENA y la regla tiene sentencia de
#    aislamiento (integridad referencial), la ejecuta.
# 3) Decide el resultado comparando el cumplimiento con el umbral.
# Una regla que falla al ejecutarse queda como ERROR y no corta
# la medición de las demás.
# ============================================================

import re

resultados = []

for regla in reglas:
    regla_id = regla["regla_id"]
    decision = regla["decision"]
    umbral = float(regla["umbral_cumplimiento"])

    evaluadas = incumplen = aisladas = None
    resultado, avisar, detalle = "ERROR", True, None
    motivo_aislamiento = None

    try:
        evaluadas = int(spark.sql(resolver(regla["consulta_evaluadas"])).collect()[0][0] or 0)
        incumplen = int(spark.sql(resolver(regla["consulta_incumplen"])).collect()[0][0] or 0)
        porcentaje = porcentaje_cumplimiento(evaluadas, incumplen)

        aisladas = 0
        sentencia = regla["consulta_cuarentena"]
        if decision == "CUARENTENA" and sentencia:
            coincide = re.search(r"motivo_cuarentena\s*=\s*'([A-Z_]+)'", sentencia)
            motivo_aislamiento = coincide.group(1) if coincide else None
            if incumplen > 0:
                aisladas = int(spark.sql(resolver(sentencia)).collect()[0]["num_updated_rows"])

        resultado, avisar = decidir(incumplen, porcentaje, umbral, decision)

        if incumplen == 0:
            detalle = "Sin filas que incumplan"
        elif resultado == "CUMPLE":
            detalle = f"{incumplen} filas incumplen, dentro de la tolerancia del umbral"
        elif decision == "CUARENTENA" and sentencia:
            detalle = f"{incumplen} filas incumplen; esta ejecución aisló {aisladas}"
        elif decision == "CUARENTENA":
            detalle = f"{incumplen} filas incumplen; ya aisladas por Silver con su motivo"
        else:
            detalle = f"{incumplen} filas incumplen"

    except Exception as e:
        porcentaje = None
        detalle = "Error al ejecutar la regla: " + str(e).splitlines()[0][:400]

    resultados.append({
        "regla_id": regla_id,
        "entidad": regla["entidad"],
        "regla_negocio": regla["regla_negocio"],
        "decision": decision,
        "umbral": umbral,
        "evaluadas": evaluadas,
        "incumplen": incumplen,
        "porcentaje": porcentaje,
        "aisladas": aisladas,
        "motivo_aislamiento": motivo_aislamiento,
        "resultado": resultado,
        "avisar": avisar,
        "detalle": detalle,
    })

    icono = {"CUMPLE": "✓", "CUARENTENA": "•", "ADVERTENCIA": "⚠️", "DETENIDO": "✗", "ERROR": "✗"}[resultado]
    print(f"{icono} {regla_id} {resultado} | evaluadas={evaluadas} incumplen={incumplen} cumplimiento={porcentaje}")

print(f"✓ Reglas medidas: {len(resultados)}")

In [0]:
# ============================================================
# Guardar resultados y conteo de cuarentena
# ------------------------------------------------------------
# El DELETE previo evita duplicar filas si la task se reintenta.
# ============================================================

spark.sql(
    f"DELETE FROM {TABLA_RESULTADOS} WHERE id_ejecucion = :id AND momento_ejecucion = :momento",
    args={"id": id_ejecucion, "momento": momento}
)

filas_resultados = [
    (id_ejecucion, r["regla_id"], momento, r["evaluadas"], r["incumplen"], r["porcentaje"], r["umbral"],
     r["decision"], r["resultado"], r["aisladas"], r["detalle"])
    for r in resultados
]

if filas_resultados:
    (
        spark.createDataFrame(
            filas_resultados,
            "id_ejecucion string, regla_id string, momento_ejecucion string, filas_evaluadas long, "
            "filas_incumplen long, porcentaje_cumplimiento double, umbral_cumplimiento double, "
            "decision_configurada string, resultado string, filas_aisladas long, detalle string"
        )
        .withColumn("porcentaje_cumplimiento", F.col("porcentaje_cumplimiento").cast("decimal(6,3)"))
        .withColumn("umbral_cumplimiento", F.col("umbral_cumplimiento").cast("decimal(5,2)"))
        .withColumn("fecha_registro", F.current_timestamp())
        .select("id_ejecucion", "regla_id", "momento_ejecucion", "filas_evaluadas", "filas_incumplen",
                "porcentaje_cumplimiento", "umbral_cumplimiento", "decision_configurada", "resultado",
                "filas_aisladas", "detalle", "fecha_registro")
        .write.format("delta").mode("append").saveAsTable(TABLA_RESULTADOS)
    )
    print(f"✓ {len(filas_resultados)} resultados guardados en {TABLA_RESULTADOS}")

# Conteo por motivo de las reglas que aíslan filas: filas con ese motivo tras esta ejecución.
for r in resultados:
    if r["motivo_aislamiento"] and r["incumplen"] is not None:
        spark.sql(
            f"DELETE FROM {TABLA_CUARENTENA_RESUMEN} "
            f"WHERE id_ejecucion = :id AND tabla_silver = :tabla AND motivo_cuarentena = :motivo",
            args={"id": id_ejecucion, "tabla": r["entidad"], "motivo": r["motivo_aislamiento"]}
        )
        spark.sql(
            f"INSERT INTO {TABLA_CUARENTENA_RESUMEN} VALUES (:id, :tabla, :motivo, :filas, current_timestamp())",
            args={"id": id_ejecucion, "tabla": r["entidad"], "motivo": r["motivo_aislamiento"], "filas": r["incumplen"]}
        )
        print(f"✓ Cuarentena registrada: {r['entidad']} · {r['motivo_aislamiento']} · {r['incumplen']} filas")

In [0]:
# ============================================================
# Resumen de la ejecución
# ============================================================

display(
    spark.table(TABLA_RESULTADOS)
    .filter((F.col("id_ejecucion") == id_ejecucion) & (F.col("momento_ejecucion") == momento))
    .orderBy("regla_id")
)

conteo = {}
for r in resultados:
    conteo[r["resultado"]] = conteo.get(r["resultado"], 0) + 1
print("✓ Resumen: " + " | ".join(f"{k}: {v}" for k, v in sorted(conteo.items())))

In [0]:
# ============================================================
# ALERTA A SLACK
# ------------------------------------------------------------
# Solo se avisa de las reglas que superaron su umbral.
# Un fallo al enviar el aviso no detiene la validación.
# ============================================================

import requests as slack_requests

con_aviso = [r for r in resultados if r["avisar"]]

if con_aviso:
    lineas = []
    for r in con_aviso:
        lineas.append(
            f"• *{r['regla_id']}* {r['resultado']} · {r['regla_negocio']} · "
            f"{r['incumplen']} de {r['evaluadas']} filas ({r['porcentaje']} % de cumplimiento, umbral {r['umbral']} %)"
            if r["incumplen"] is not None else
            f"• *{r['regla_id']}* {r['resultado']} · {r['detalle']}"
        )

    try:
        webhook_url = dbutils.secrets.get(scope="alerts", key="slack_webhook_url")
        slack_payload = {
            "text": (
                f":warning: *Calidad de datos DataCo* ({momento})\n"
                f"*Ejecución:* {id_ejecucion}\n" + "\n".join(lineas)
            )
        }
        slack_response = slack_requests.post(webhook_url, json=slack_payload, timeout=10)
        if slack_response.status_code == 200:
            print(f"✓ Aviso enviado a Slack ({len(con_aviso)} reglas)")
        else:
            print(f"⚠️ Slack respondió con status {slack_response.status_code}")
    except Exception as ex:
        print(f"⚠️ No se pudo enviar el aviso a Slack: {ex}")
else:
    print("✓ Ninguna regla superó su umbral: no se envía aviso")

In [0]:
# ============================================================
# Decisión final y cierre del log
# ------------------------------------------------------------
# Si alguna regla de DETENER incumple, o alguna regla no se pudo
# ejecutar, la task falla DESPUÉS de haber guardado los resultados
# y enviado el aviso. El log de error de la task marca el registro.
# ============================================================

fallos = [r for r in resultados if r["resultado"] in ("DETENIDO", "ERROR")]

if fallos:
    detalle_fallos = " | ".join(f"{r['regla_id']} {r['resultado']}: {r['detalle']}" for r in fallos)
    raise Exception(f"Calidad de datos ({momento}): {len(fallos)} regla(s) detienen el pipeline. {detalle_fallos}")

filas_procesadas = len(resultados)

spark.sql(f"""
UPDATE {TABLA_ETL_LOG}
SET estado = 'EXITOSO',
    fecha_fin = current_timestamp(),
    filas_procesadas = {filas_procesadas}
WHERE log_id = '{LOG_ID}'
""")

print(f"✓ etl_log actualizado: EXITOSO ({filas_procesadas} reglas evaluadas)")